
# 🎰 Lab: Học tăng cường với Blackjack – Q-learning  
**Môn:** Thuật toán ứng dụng  
**Ngôn ngữ:** Python (Gymnasium, NumPy, Matplotlib)  
**Mục tiêu:** Hiểu và triển khai thuật toán Q-learning trên môi trường Blackjack-v1.

---
## 📘 Bài 1 – Tìm hiểu môi trường Blackjack

**Yêu cầu:**  
- Tạo môi trường `Blackjack-v1 (sab=True)`  
- In ra ví dụ trạng thái, hành động, phần thưởng  
- Giải thích ý nghĩa các biến quan sát

**Gợi ý:** Quan sát là bộ 3 `(player_sum, dealer_card, usable_ace)`.


In [ ]:

import gymnasium as gym

env = gym.make("Blackjack-v1", sab=True)
state, _ = env.reset()
print("Trạng thái ban đầu:", state)

action = env.action_space.sample()
next_state, reward, terminated, truncated, info = env.step(action)
print(f"Thực hiện hành động {action}, thu được reward={reward}, next_state={next_state}")



---
## 🧮 Bài 2 – Huấn luyện Q-learning

**Yêu cầu:**  
- Cài đặt Q-learning với ε-greedy  
- Tham số: `alpha=0.1`, `gamma=0.9`, `epsilon=0.1`  
- Vẽ đồ thị phần thưởng trung bình theo số episode


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

env = gym.make("Blackjack-v1", sab=True)
Q = {}

def epsilon_greedy(state, eps=0.1):
    if np.random.rand() < eps or state not in Q:
        return env.action_space.sample()
    return np.argmax(Q[state])

def update_Q(state, action, reward, next_state, done, alpha=0.1, gamma=0.9):
    if state not in Q:
        Q[state] = np.zeros(env.action_space.n)
    if next_state not in Q:
        Q[next_state] = np.zeros(env.action_space.n)
    target = reward + (0 if done else gamma * np.max(Q[next_state]))
    Q[state][action] += alpha * (target - Q[state][action])

rewards = []
for episode in range(100000):
    state, _ = env.reset()
    total_reward = 0
    done = False
    while not done:
        action = epsilon_greedy(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        update_Q(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
    rewards.append(total_reward)

plt.plot(np.convolve(rewards, np.ones(1000)/1000, mode='valid'))
plt.xlabel("Episode")
plt.ylabel("Trung bình reward (cửa sổ 1000)")
plt.title("Đường học Q-learning Blackjack")
plt.show()



---
## 🔍 Bài 3 – Phân tích siêu tham số

So sánh hai giá trị `epsilon = 0.1` và `0.3`.  
Nhận xét sự khác biệt về tốc độ hội tụ và tỉ lệ thắng.


In [ ]:

def train_qlearning(epsilon, episodes=50000):
    Q = {}
    env = gym.make("Blackjack-v1", sab=True)
    rewards = []
    for ep in range(episodes):
        state, _ = env.reset()
        total = 0
        done = False
        while not done:
            if state not in Q:
                Q[state] = np.zeros(env.action_space.n)
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                action = np.argmax(Q[state])
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            if next_state not in Q:
                Q[next_state] = np.zeros(env.action_space.n)
            Q[state][action] += 0.1 * (reward + 0.9*np.max(Q[next_state]) - Q[state][action])
            state = next_state
            total += reward
        rewards.append(total)
    return np.convolve(rewards, np.ones(500)/500, mode='valid')

r1 = train_qlearning(0.1)
r2 = train_qlearning(0.3)
plt.plot(r1, label="ε=0.1")
plt.plot(r2, label="ε=0.3")
plt.legend()
plt.title("Ảnh hưởng của epsilon đến tốc độ học")
plt.show()



---
## ⚖️ Bài 4 – So sánh Q-learning và SARSA
Cài đặt thêm thuật toán SARSA và vẽ hai đường học trên cùng biểu đồ.


In [ ]:

def train_sarsa(episodes=50000, epsilon=0.1):
    Q = {}
    env = gym.make("Blackjack-v1", sab=True)
    rewards = []
    for ep in range(episodes):
        state, _ = env.reset()
        if state not in Q: Q[state] = np.zeros(env.action_space.n)
        action = np.argmax(Q[state]) if np.random.rand() > epsilon else env.action_space.sample()
        done = False
        total = 0
        while not done:
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            if next_state not in Q: Q[next_state] = np.zeros(env.action_space.n)
            next_action = np.argmax(Q[next_state]) if np.random.rand() > epsilon else env.action_space.sample()
            Q[state][action] += 0.1 * (reward + 0.9*Q[next_state][next_action]*(not done) - Q[state][action])
            state, action = next_state, next_action
            total += reward
        rewards.append(total)
    return np.convolve(rewards, np.ones(500)/500, mode='valid')

r_q = train_qlearning(0.1)
r_sarsa = train_sarsa()
plt.plot(r_q, label="Q-learning")
plt.plot(r_sarsa, label="SARSA")
plt.legend()
plt.title("So sánh Q-learning vs SARSA")
plt.show()



---
## 🎨 Bài 5 – Trực quan chính sách học được

Vẽ heatmap chính sách (hit/stand) theo `(player_sum, dealer_card)` cho trường hợp có usable ace.


In [ ]:

import seaborn as sns
Q_learned = Q  # Dùng Q từ huấn luyện trước đó
policy = np.zeros((22, 11))
for psum in range(12, 22):
    for dcard in range(1, 11):
        state = (psum, dcard, True)
        if state in Q_learned:
            policy[psum, dcard] = np.argmax(Q_learned[state])
sns.heatmap(policy[12:22, 1:11], cmap="YlGnBu", cbar=False)
plt.xlabel("Lá bài ngửa của nhà cái")
plt.ylabel("Tổng điểm người chơi")
plt.title("Chính sách học được (1=Hit, 0=Stand)")
plt.show()



---
### 📄 Ghi chú nộp bài
- Nộp file `.ipynb` sau khi hoàn thành 5 bài.
- Mỗi bài có ít nhất 1 ô Markdown mô tả kết quả.
- Có thể mở rộng thêm thử nghiệm nếu muốn (tăng số episode, thử epsilon khác, v.v.).
